# Unlearning vs. Membership Inference: all experiments

Runs every target model of Table 1 (1a ... 3d) with every unlearning method (NegGrad, SCRUB, SFTC) with the unlearning learning rates selected by the learning-rate grid (`LEARNING_RATES`), and fills the paper table `tab:utility_mia_all`. All MIAs are evaluated after every unlearning epoch on the whole evaluation data.

**What happens for each target model**
1. The data splits used to train the saved models are rebuilt (seed 42, same split functions as `train_target_and_perform_MIA_attack.py`).
2. The saved models are loaded from `models/`: target, retrained-from-scratch baseline, and the shadow-model attack (MIA-1). Nothing is retrained.
3. Each method unlearns the target for `EPOCHS` epochs. After **every** epoch it measures test accuracy and the forget/retain AUC of
   - **MIA-1**: shadow-model + RandomForest attack (saved `.jolib` model)
   - **MIA-2**: loss-threshold attack
   - **MIA-3**: offline **LiRA** (Carlini et al., 2022), with 16 reference models per target model (Step 1)
4. **Best** = epoch minimising `|MIA-1 forget AUC − Retr.| + |Test Acc − Retr. Test Acc|`; **Final** = last epoch. AUC is reported as max(AUC, 1−AUC).

**Reproducibility**
- Fixed seeds, deterministic cuDNN/cuBLAS, every run reseeded at its start, so results don't depend on run order.
- Each target model runs in its own subprocess (clean state, and memory is freed between experiments).
- Results are cached in `results_lira/<exp>/<method>.json` together with their config and software versions. Re-running the notebook skips finished runs, so it **resumes after an interruption**. Changing a learning rate (or epochs/seed) re-runs only that run. To force a re-run, delete its JSON file.

**Run it without keeping the notebook open** (from the project folder):
```
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 run_all_experiments.ipynb
```
Outputs: `results/summary.csv`, `results/table_utility_mia_all.tex`, and per-epoch curves in `results/<exp>/<method>_epochs.csv`.

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # deterministic cuBLAS; set before CUDA starts

import json
import pandas as pd
import experiment_runner as er

env = er.environment_info()
print(json.dumps(env, indent=2))
if env["uncommitted_python_changes"]:
    print("Note: there are uncommitted .py changes; commit them so results can be tied to a git commit.")

/home/theo/miniconda3/envs/unlearn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "python": "3.11.16",
  "torch": "2.0.1+cu117",
  "cuda": "11.7",
  "cudnn": 8500,
  "gpu": "NVIDIA GeForce RTX 3050 Laptop GPU",
  "numpy": "1.23.2",
  "pandas": "1.5.3",
  "sklearn": "1.5.1",
  "git_commit": "e56dcccd",
  "uncommitted_python_changes": false,
  "runner_version": 1
}


## Configuration

`LEARNING_RATES`: one unlearning learning rate per target model and method, selected by the learning-rate grid (`results_lrgrid/table_lr_grid.tex`). Set a value to `None` to skip that run (it shows as `--` in the table). Changing a value re-runs only that run.

In [2]:
SEED = 42
EPOCHS = 30          # unlearning epochs
BATCH_SIZE = 32
MOMENTUM = 0.9       # SGD momentum, as in main.py
RESULTS_DIR = "results_lira"   # results/ holds the previous runs (MIA-3 = max confidence)
MIA3 = "lira"                  # MIA-3 = offline LiRA

EXPERIMENTS_TO_RUN = er.ALL_EXPERIMENTS   # e.g. ["1d", "2d", "3d"] to run a subset
METHODS = er.METHODS                       # ["neg_grad", "scrub", "sftc"]

LEARNING_RATES = {   # selected by the learning-rate grid (results_lrgrid/table_lr_grid.tex)
    "1a": {"neg_grad": 1e-3, "scrub": 1e-5, "sftc": 1e-3},
    "1b": {"neg_grad": 1e-2, "scrub": 1e-4, "sftc": 1e-2},   # NegGrad: best at epoch 1, diverges after
    "1c": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-2},
    "1d": {"neg_grad": 1e-4, "scrub": 1e-3, "sftc": 1e-2},
    "2a": {"neg_grad": 1e-4, "scrub": 1e-5, "sftc": 1e-3},
    "2b": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-3},
    "2c": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-3},
    "2d": {"neg_grad": 1e-3, "scrub": 1e-4, "sftc": 1e-2},
    "3a": {"neg_grad": 1e-4, "scrub": 1e-4, "sftc": 1e-2},   # NegGrad: selected model collapsed (test acc 0.15)
    "3b": {"neg_grad": 1e-4, "scrub": 1e-3, "sftc": 1e-2},
    "3c": {"neg_grad": 1e-4, "scrub": 1e-7, "sftc": 1e-2},   # SCRUB: flat region, no LR changes the model
    "3d": {"neg_grad": 1e-4, "scrub": 1e-6, "sftc": 1e-2},   # SCRUB: flat region, no LR changes the model
}

missing = [(e, er.METHOD_NAMES[m]) for e in EXPERIMENTS_TO_RUN for m in METHODS if LEARNING_RATES[e][m] is None]
print(f"{len(missing)} runs have no learning rate yet and will be skipped:")
print(", ".join(f"{e} {m}" for e, m in missing) or "none")

0 runs have no learning rate yet and will be skipped:
none


## Step 1: LiRA reference models (for MIA-3)
For every target model, 16 reference models of the same architecture are trained on random subsets of the shadow data (disjoint from the target data), with the target's training settings (`lira.TRAIN_CONFIG`). Only missing models are trained; each one is saved as soon as it is trained (`models/lira/<exp>/`). The tabular models take minutes, the ResNets (3a, 3b) hours.

In [3]:
import lira
import train_lira_references as tlr

failed_lira = tlr.run(EXPERIMENTS_TO_RUN, lira.N_REFERENCE)

Files already downloaded and verified
[1a] TargetModel_1a | shadow 20000, reference train size 16000 | pools: forget 3200, retain 12800, non_member 10000 | config {'lr': 0.001, 'epochs': 300, 'batch_size': 32, 'momentum': 0.9, 'stop_train_acc': 0.999} | loaded in 14s
  [1a] reference 10/16: 56 epochs, train acc 1.000, held-out shadow acc 0.472, 57s
  [1a] reference 11/16: 60 epochs, train acc 0.999, held-out shadow acc 0.474, 60s
  [1a] reference 12/16: 56 epochs, train acc 1.000, held-out shadow acc 0.474, 57s
  [1a] reference 13/16: 56 epochs, train acc 1.000, held-out shadow acc 0.483, 57s
  [1a] reference 14/16: 54 epochs, train acc 0.999, held-out shadow acc 0.481, 54s
  [1a] reference 15/16: 58 epochs, train acc 0.999, held-out shadow acc 0.469, 58s
  [1a] reference 16/16: 59 epochs, train acc 0.999, held-out shadow acc 0.495, 59s
[1b] TargetModel_1a | shadow 2406, reference train size 1924 | pools: forget 385, retain 1539, non_member 1203 | config {'lr': 0.001, 'epochs': 300, 'b

## Step 2: run all experiments
One line per epoch is printed. Finished runs are loaded from `results_lira/` instead of re-run.

In [4]:
failed = er.run_all(EXPERIMENTS_TO_RUN, METHODS, LEARNING_RATES, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    momentum=MOMENTUM, seed=SEED, results_dir=RESULTS_DIR, mia3=MIA3)

Files already downloaded and verified
[1a] CIFAR-10 | TargetModel_1a | retain 12800, forget 3200, test 4000, MIA sample 3200 | loaded in 15s
  Original : test acc 0.489, forget acc 1.000, retain acc 1.000, MIA-1 forget AUC 0.848
  Retrained: test acc 0.478, forget acc 0.478, retain acc 1.000, MIA-1 forget AUC 0.501
  [1a NegGrad lr=0.001] epoch  1/30 | test 0.489 | forget AUC MIA-1 0.852 MIA-2 0.856 MIA-3 0.873 | MIA-1 acc forget 0.803 retain 0.799 | 1s
  [1a NegGrad lr=0.001] epoch  2/30 | test 0.489 | forget AUC MIA-1 0.853 MIA-2 0.856 MIA-3 0.873 | MIA-1 acc forget 0.803 retain 0.802 | 2s
  [1a NegGrad lr=0.001] epoch  3/30 | test 0.489 | forget AUC MIA-1 0.851 MIA-2 0.856 MIA-3 0.873 | MIA-1 acc forget 0.802 retain 0.808 | 3s
  [1a NegGrad lr=0.001] epoch  4/30 | test 0.491 | forget AUC MIA-1 0.839 MIA-2 0.855 MIA-3 0.871 | MIA-1 acc forget 0.790 retain 0.796 | 4s
  [1a NegGrad lr=0.001] epoch  5/30 | DIVERGED (weights are inf/NaN). Stopping this run; try a smaller learning rate.
 

## Results

In [9]:
RESULTS_DIR = 'results'

results = er.load_results(RESULTS_DIR)
summary = er.summary_dataframe(results)
summary.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
summary.round(3)

,exp,dataset,method,lr,best_epoch,diverged_at_epoch,test_acc_start,forget_mia1_auc_start,forget_mia2_auc_start,forget_mia3_auc_start,...,forget_mia2_auc_retrain,forget_mia3_auc_retrain,test_acc_best,forget_mia1_auc_best,forget_mia2_auc_best,forget_mia3_auc_best,test_acc_final,forget_mia1_auc_final,forget_mia2_auc_final,forget_mia3_auc_final
0,1a,CIFAR-10,NegGrad,0.001,4,5.0,0.489,0.848,0.855,0.806,...,0.505,0.502,0.491,0.839,0.855,0.803,NaN,NaN,NaN,NaN
1,1a,CIFAR-10,SCRUB,0.000,26,NaN,0.489,0.848,0.855,0.806,...,0.505,0.502,0.489,0.847,0.855,0.806,0.489,0.848,0.855,0.806
2,1a,CIFAR-10,SFTC,0.001,9,NaN,0.489,0.848,0.855,0.806,...,0.505,0.502,0.487,0.504,0.725,0.546,0.464,0.663,0.638,0.690
3,1b,MuFac,NegGrad,0.010,1,2.0,0.336,0.911,0.943,0.895,...,0.523,0.503,0.307,0.606,0.832,0.643,NaN,NaN,NaN,NaN
4,1b,MuFac,SCRUB,0.000,5,NaN,0.336,0.911,0.943,0.895,...,0.523,0.503,0.301,0.509,0.552,0.522,0.278,0.502,0.514,0.524
5,1b,MuFac,SFTC,0.010,30,NaN,0.336,0.911,0.943,0.895,...,0.523,0.503,0.320,0.501,0.536,0.507,0.320,0.501,0.536,0.507
6,1c,Purchase-100,NegGrad,0.001,1,10.0,0.800,0.778,0.717,0.710,...,0.502,0.503,0.776,0.737,0.693,0.680,NaN,NaN,NaN,NaN
7,1c,Purchase-100,SCRUB,0.000,28,NaN,0.800,0.778,0.717,0.710,...,0.502,0.503,0.799,0.777,0.717,0.710,0.800,0.777,0.717,0.710
8,1c,Purchase-100,SFTC,0.010,22,NaN,0.800,0.778,0.717,0.710,...,0.502,0.503,0.654,0.505,0.572,0.502,0.640,0.510,0.551,0.531
9,1d,Texas-100,NegGrad,0.000,11,NaN,0.578,0.573,0.683,0.601,...,0.504,0.507,0.569,0.570,0.675,0.591,0.046,0.500,0.501,0.500


## LaTeX table
Written to `results_lira/table_utility_mia_all.tex`; include it with `\input{...}`. Final/Best test accuracies below 25% of the retrained model's are bolded (model collapse).

In [13]:
RESULTS_DIR = 'results'
tex = er.make_latex_table(results)
with open(os.path.join(RESULTS_DIR, "table_utility_mia_all.tex"), "w") as f:
    f.write(tex)
print(tex)

% Requires: \usepackage{booktabs,multirow,graphicx,rotating}
% Generated by experiment_runner.make_latex_table. Rows: Dataset -> Target model ID (Table 1) -> method.
% Start = original model, Retr. = retrained from scratch on the retain set (merged cells).
% Best epoch = argmin |MIA-1 forget AUC - Retr.| + |Test Acc - Retr.|. '--' = not run yet.

\providecommand{\startcell}[1]{\multirow{3}{*}{#1}}

\begin{sidewaystable}
\centering
\caption{Test accuracy and forget-set MIA AUC for all 12 target models (Table~1) and all three
unlearning methods: the original model before unlearning (Start), the model retrained from scratch
on the retain set (Retr.), the unlearned model at the epoch closest to the retrained model (Best;
epoch in parentheses), and after the last unlearning epoch (Final). Start and Retr. do not depend on
the unlearning method and are therefore given once per target model. MIA-2 and MIA-3 are evaluated at
the same epochs as MIA-1. AUC is reported as $\max(\mathrm{AUC}, 1-\ma

## Optional: per-epoch curves for one run

In [7]:
import matplotlib.pyplot as plt

EXP, METHOD = "3c", "sftc"   # pick any finished run
path = os.path.join(RESULTS_DIR, EXP, f"{METHOD}_epochs.csv")
if os.path.exists(path):
    h, r = pd.read_csv(path), results[EXP][METHOD]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["epoch"], h["test_acc"], label="Test acc")
    axes[0].axhline(r["retrain"]["test_acc"], ls="--", c="gray", label="Retrained")
    for a, c in zip(er.ATTACKS, ["C0", "C1", "C2"]):
        axes[1].plot(h["epoch"], h[f"forget_{a}_auc"], c=c, label={"mia1": "MIA-1", "mia2": "MIA-2", "mia3": "MIA-3 (LiRA)"}[a])
        axes[1].axhline(r["retrain"][f"forget_{a}_auc"], ls="--", c=c)
    for ax, t in zip(axes, ["Test accuracy", "Forget-set MIA AUC (dashed: retrained)"]):
        ax.axvline(r["best_epoch"], c="k", ls=":", label="Best epoch")
        ax.set_title(t); ax.set_xlabel("Unlearning epoch"); ax.legend(); ax.grid(alpha=.3)
    fig.suptitle(f"{EXP} {er.METHOD_NAMES[METHOD]} (lr={r['config']['lr']:g})")
    plt.show()
else:
    print(f"No results yet for {EXP} {METHOD}")

<Figure size 1200x400 with 2 Axes>